# Data Exploration - Billups Challenge
Quick exploration of the datasets before writing the actual solution

In [ ]:
# install pyspark on colab
!pip install pyspark -q

In [ ]:
# upload files - run this cell and use the dialog to upload
# historical_transactions.parquet and merchants.csv
from google.colab import files
uploaded = files.upload()

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("billups-exploration")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("spark ready")

## 1. Load raw data

In [ ]:
transactions = spark.read.parquet("historical_transactions.parquet")
merchants = spark.read.csv("merchants.csv", header=True, inferSchema=True)

print(f"transactions rows: {transactions.count()}")
print(f"merchants rows: {merchants.count()}")

## 2. Schemas - what columns do we actually have?

In [ ]:
print("=== TRANSACTIONS SCHEMA ===")
transactions.printSchema()
print(f"\nColumns: {transactions.columns}")

In [ ]:
print("=== MERCHANTS SCHEMA ===")
merchants.printSchema()
print(f"\nColumns: {merchants.columns}")

## 3. Sample data - see what we're working with

In [ ]:
transactions.show(10, truncate=False)

In [ ]:
merchants.show(10, truncate=False)

## 4. Nulls check - important for cleaning rules

In [ ]:
# null counts for transactions
print("=== TRANSACTIONS NULL COUNTS ===")
trans_nulls = transactions.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in transactions.columns]
)
trans_nulls.show(truncate=False)

In [ ]:
# null counts for merchants
print("=== MERCHANTS NULL COUNTS ===")
merch_nulls = merchants.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in merchants.columns]
)
merch_nulls.show(truncate=False)

## 5. Key columns - unique values and distributions

In [ ]:
# how many unique merchants, cities, states, categories
print("=== TRANSACTIONS UNIQUES ===")
print(f"unique merchant_id: {transactions.select('merchant_id').distinct().count()}")
print(f"unique city_id: {transactions.select('city_id').distinct().count()}")
print(f"unique state_id: {transactions.select('state_id').distinct().count()}")
print(f"unique category: {transactions.select('category').distinct().count()}")

In [ ]:
# what categories exist?
print("=== CATEGORIES ===")
transactions.groupBy("category").count().orderBy(F.desc("count")).show(20)

In [ ]:
# unique merchants in merchants table
print(f"unique merchant_id in merchants: {merchants.select('merchant_id').distinct().count()}")

## 6. Verify the join - do all transactions have a matching merchant?

In [ ]:
# merchant_ids in transactions that are NOT in merchants table
trans_merchants = transactions.select("merchant_id").distinct()
merch_merchants = merchants.select("merchant_id").distinct()

orphan_trans = trans_merchants.subtract(merch_merchants)
print(f"transaction merchant_ids with NO match in merchants: {orphan_trans.count()}")

# merchants with no transactions
orphan_merch = merch_merchants.subtract(trans_merchants)
print(f"merchant_ids with NO transactions: {orphan_merch.count()}")

## 7. Date range and time analysis - for Q1 and Q3

In [ ]:
# check the date column type and sample values
transactions.select("purchase_date").show(5, truncate=False)
print(f"purchase_date dtype: {dict(transactions.dtypes)['purchase_date']}")

In [ ]:
# date range
transactions.select(
    F.min("purchase_date").alias("earliest"),
    F.max("purchase_date").alias("latest")
).show(truncate=False)

In [ ]:
# how many months of data do we have?
transactions.select(
    F.date_format("purchase_date", "MMM yyyy").alias("month")
).distinct().orderBy("month").show(50)

In [ ]:
# hour distribution - can we extract hours?
transactions.select(
    F.hour("purchase_date").alias("hour")
).groupBy("hour").count().orderBy("hour").show(24)

## 8. purchase_amount - basic stats

In [ ]:
transactions.select("purchase_amount").describe().show()

# check for negatives or zeros
print(f"negative amounts: {transactions.filter(F.col('purchase_amount') < 0).count()}")
print(f"zero amounts: {transactions.filter(F.col('purchase_amount') == 0).count()}")

## 9. Installments - for Q5e

In [ ]:
# what values does installments have?
transactions.groupBy("installments").agg(
    F.count("*").alias("num_transactions"),
    F.sum("purchase_amount").alias("total_amount"),
    F.avg("purchase_amount").alias("avg_amount")
).orderBy("installments").show(20)

In [ ]:
# installments = 1 vs installments > 1 comparison
transactions.withColumn(
    "is_installment", F.when(F.col("installments") > 1, "yes").otherwise("no")
).groupBy("is_installment").agg(
    F.count("*").alias("transactions"),
    F.sum("purchase_amount").alias("total_amount"),
    F.avg("purchase_amount").alias("avg_amount")
).show()

## 10. City distribution - for Q4 and Q5a

In [ ]:
# top 20 cities by transaction count
transactions.groupBy("city_id").agg(
    F.count("*").alias("num_transactions"),
    F.sum("purchase_amount").alias("total_amount")
).orderBy(F.desc("num_transactions")).show(20)

In [ ]:
# city vs category crosstab - is there a pattern?
transactions.groupBy("city_id", "category").count().orderBy(
    F.desc("count")
).show(30)

## 11. Monthly trends - for Q5c

In [ ]:
# sales by month
transactions.withColumn(
    "year_month", F.date_format("purchase_date", "yyyy-MM")
).groupBy("year_month").agg(
    F.count("*").alias("num_transactions"),
    F.sum("purchase_amount").alias("total_amount")
).orderBy("year_month").show(24)

## 12. Quick sanity check - merchant_name values

In [ ]:
# are merchant_names actually anonymized strings?
merchants.select("merchant_name").show(10, truncate=False)

# any null merchant_names?
print(f"null merchant_names: {merchants.filter(F.col('merchant_name').isNull()).count()}")

In [ ]:
# check if merchant_id is truly unique in merchants table
total = merchants.count()
unique = merchants.select("merchant_id").distinct().count()
print(f"total merchants: {total}")
print(f"unique merchant_ids: {unique}")
print(f"duplicates: {total - unique}")

## 13. State distribution - for Q2

In [ ]:
# how many states, and distribution
transactions.groupBy("state_id").agg(
    F.count("*").alias("transactions"),
    F.countDistinct("merchant_id").alias("unique_merchants"),
    F.avg("purchase_amount").alias("avg_amount")
).orderBy(F.desc("transactions")).show(20)

## Summary
After running all cells, check:
- [ ] What columns exist in both tables (any surprises?)
- [ ] How many orphan merchant_ids (transactions without merchant match)
- [ ] Date format - is purchase_date a timestamp or string?
- [ ] Can we extract hours from purchase_date?
- [ ] How many null categories exist
- [ ] Are there duplicate merchant_ids in merchants table?
- [ ] What does the installments column look like
- [ ] Any negative or weird purchase_amounts